### import libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from scipy.sparse import load_npz
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense,Input,Embedding,SimpleRNN,LSTM,GRU,Bidirectional)
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score)

I0000 00:00:1787475760.418679  598237 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787475760.521870  598237 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787475764.399050  598237 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Load the datasets

In [2]:
# Phase 2: padded sequences
X_train_padded = np.load("../models/X_train_padded.npy")
X_val_padded = np.load("../models/X_val_padded.npy")
X_test_padded = np.load("../models/X_test_padded.npy")

# Phase 3: TF-IDF
X_train_tfidf = load_npz("../models/X_train_tfidf.npz")
X_val_tfidf = load_npz("../models/X_val_tfidf.npz")
X_test_tfidf = load_npz("../models/X_test_tfidf.npz")

# Labels
y_train = np.load("../models/y_train.npy")
y_val = np.load("../models/y_val.npy")
y_test = np.load("../models/y_test.npy")

In [3]:

print("X_train:", X_train_padded.shape)
print("X_val:", X_val_padded.shape)
print("X_test:", X_test_padded.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)


print("X_train:", X_train_tfidf.shape)
print("X_val:", X_val_tfidf.shape)
print("X_test:", X_test_tfidf.shape)



X_train: (34705, 200)
X_val: (7439, 200)
X_test: (7438, 200)
y_train: (34705,)
y_val: (7439,)
y_test: (7438,)
X_train: (34705, 20000)
X_val: (7439, 20000)
X_test: (7438, 20000)


### Load the Tokenizer

In [4]:
with open("../models/tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary size:", vocab_size)

Vocabulary size: 85870


In [ ]:
sequence_length = X_train_padded.shape[1]
print("Sequence length:", sequence_length)

Sequence length: 200


### ANN model

In [60]:
ann_model = Sequential([
    Input(shape=(X_train_tfidf.shape[1],)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [61]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
ann_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (None, 128)            │     2,560,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,568,449 (9.80 MB)

 Trainable params: 2,568,449 (9.80 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_ann = ann_model.fit(X_train_tfidf,y_train,validation_data=(X_val_tfidf, y_val),epochs=10,batch_size=32)

In [ ]:
print(history_ann.history.keys())

In [7]:
results = []

## ANN Predictions

In [8]:
y_pred_prob_ann = ann_model.predict(X_test_tfidf)

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step


In [9]:
y_pred_ann = (y_pred_prob_ann >= 0.5).astype(int).ravel()

In [ ]:
ann_accuracy = accuracy_score(y_test, y_pred_ann)
ann_precision = precision_score(y_test,y_pred_ann)
ann_recall = recall_score(y_test,y_pred_ann)
ann_f1 = f1_score(y_test,y_pred_ann)
ann_roc_auc = roc_auc_score(y_test,y_pred_prob_ann.ravel())

In [11]:
print("ANN Results")
print("----------------------")
print("Accuracy :", ann_accuracy)
print("Precision:", ann_precision)
print("Recall   :", ann_recall)
print("F1 Score :", ann_f1)
print("ROC-AUC  :", ann_roc_auc)

ANN Results
----------------------
Accuracy : 0.8710674912610917
Precision: 0.8756771397616468
Recall   : 0.8660594695954996
F1 Score : 0.8708417508417509
ROC-AUC  : 0.9430450882507222


In [ ]:
ann_model.save("../models/ann_model.keras")

In [12]:
results.append({
    "Model": "ANN",
    "Accuracy": ann_accuracy,
    "Precision": ann_precision,
    "Recall": ann_recall,
    "F1 Score": ann_f1,
    "ROC-AUC": ann_roc_auc
})

### RNN Model

In [ ]:
rnn_model = Sequential([
    Input(shape=(sequence_length,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    
    SimpleRNN(64),
    
    Dense(1, activation="sigmoid")
])
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
rnn_model.summary()

In [ ]:
history_rnn = rnn_model.fit(X_train_padded,y_train,validation_data=(X_val_padded, y_val),epochs=10,batch_size=32)

#### RNN Prediction

In [ ]:
y_pred_prob_rnn = rnn_model.predict(X_test_padded)
y_pred_rnn = (y_pred_prob_rnn >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step


In [ ]:
rnn_accuracy = accuracy_score(y_test, y_pred_rnn)
rnn_precision = precision_score(y_test,y_pred_rnn)
rnn_recall = recall_score(y_test,y_pred_rnn)
rnn_f1 = f1_score(y_test,y_pred_rnn)
rnn_roc_auc = roc_auc_score(y_test,y_pred_prob_rnn.ravel())

In [15]:
print("RNN Results")
print("----------------------")
print("Accuracy :", rnn_accuracy)
print("Precision:", rnn_precision)
print("Recall   :", rnn_recall)
print("F1 Score :", rnn_f1)
print("ROC-AUC  :", rnn_roc_auc)

RNN Results
----------------------
Accuracy : 0.4989244420543157
Precision: 0.5008055853920516
Recall   : 0.4995981784087865
F1 Score : 0.5002011532787984
ROC-AUC  : 0.5021881652967135


In [ ]:
rnn_model.save("../models/rnn_model.keras")

In [16]:
results.append({
    "Model": "RNN",
    "Accuracy": rnn_accuracy,
    "Precision": rnn_precision,
    "Recall": rnn_recall,
    "F1 Score": rnn_f1,
    "ROC-AUC": rnn_roc_auc
})

### LSTM Model

In [ ]:
lstm_model = Sequential([
    Input(shape=(sequence_length,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    
    LSTM(64),
    
    Dense(1, activation="sigmoid")
])
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
lstm_model.summary()

In [ ]:
history_lstm = lstm_model.fit(X_train_padded,y_train,validation_data=(X_val_padded, y_val),epochs=10,batch_size=32)

### LSTM Prediction

In [ ]:
y_pred_prob_lstm = lstm_model.predict(X_test_padded)
y_pred_lstm = (y_pred_prob_lstm >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step


In [ ]:
lstm_accuracy = accuracy_score(y_test,y_pred_lstm)
lstm_precision = precision_score(y_test,y_pred_lstm)
lstm_recall = recall_score(y_test,y_pred_lstm)
lstm_f1 = f1_score(y_test,y_pred_lstm)
lstm_roc_auc = roc_auc_score(y_test,y_pred_prob_lstm.ravel())

In [19]:
print("LSTM Results")
print("----------------------")
print("Accuracy :", lstm_accuracy)
print("Precision:", lstm_precision)
print("Recall   :", lstm_recall)
print("F1 Score :", lstm_f1)
print("ROC-AUC  :", lstm_roc_auc)

LSTM Results
----------------------
Accuracy : 0.8631352514116698
Precision: 0.8624833110814419
Recall   : 0.8652558264130726
F1 Score : 0.8638673442096817
ROC-AUC  : 0.9317602822403532


In [ ]:
lstm_model.save("../models/lstm_model.keras")

In [20]:
results.append({
    "Model": "LSTM",
    "Accuracy": lstm_accuracy,
    "Precision": lstm_precision,
    "Recall": lstm_recall,
    "F1 Score": lstm_f1,
    "ROC-AUC": lstm_roc_auc
})

### GRU Model

In [ ]:
gru_model = Sequential([
    Input(shape=(sequence_length,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    
    GRU(64),
    
    Dense(1, activation="sigmoid")
])
gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
gru_model.summary()

In [ ]:
history_gru = gru_model.fit(X_train_padded,y_train,validation_data=(X_val_padded, y_val),epochs=10,batch_size=32)

### GRU Prediction

In [ ]:
y_pred_prob_gru = gru_model.predict(X_test_padded)
y_pred_gru = (y_pred_prob_gru >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step


In [ ]:
gru_accuracy = accuracy_score(y_test,y_pred_gru)
gru_precision = precision_score(y_test,y_pred_gru)
gru_recall = recall_score(y_test,y_pred_gru)
gru_f1 = f1_score(y_test,y_pred_gru)
gru_roc_auc = roc_auc_score(y_test,y_pred_prob_gru.ravel())

In [23]:
print("GRU Results")
print("----------------------")
print("Accuracy :", gru_accuracy)
print("Precision:", gru_precision)
print("Recall   :", gru_recall)
print("F1 Score :", gru_f1)
print("ROC-AUC  :", gru_roc_auc)

GRU Results
----------------------
Accuracy : 0.8707986017746706
Precision: 0.8666666666666667
Recall   : 0.8775783552102866
F1 Score : 0.8720883801410888
ROC-AUC  : 0.9388268110983016


In [ ]:
gru_model.save("../models/gru_model.keras")

In [24]:
results.append({
    "Model": "GRU",
    "Accuracy": gru_accuracy,
    "Precision": gru_precision,
    "Recall": gru_recall,
    "F1 Score": gru_f1,
    "ROC-AUC": gru_roc_auc
})

### Bidirectional LSTM

In [ ]:
bilstm_model = Sequential([
    Input(shape=(sequence_length,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    
    Bidirectional(
        LSTM(64)
    ),
    
    Dense(1, activation="sigmoid")
])
bilstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
bilstm_model.summary()

In [ ]:
history_bilstm = bilstm_model.fit(X_train_padded,y_train,validation_data=(X_val_padded, y_val),epochs=10,batch_size=32)

### Bidirectional LSTM

In [ ]:
y_pred_prob_bilstm = bilstm_model.predict(X_test_padded)
y_pred_bilstm = (y_pred_prob_bilstm >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 10s 40ms/step


In [ ]:
bilstm_accuracy = accuracy_score(y_test,y_pred_bilstm)
bilstm_precision = precision_score(y_test,y_pred_bilstm)
bilstm_recall = recall_score(y_test,y_pred_bilstm)
bilstm_f1 = f1_score(y_test,y_pred_bilstm)
bilstm_roc_auc = roc_auc_score(y_test,y_pred_prob_bilstm.ravel())

In [27]:
print("Bi-LSTM Results")
print("----------------------")
print("Accuracy :", bilstm_accuracy)
print("Precision:", bilstm_precision)
print("Recall   :", bilstm_recall)
print("F1 Score :", bilstm_f1)
print("ROC-AUC  :", bilstm_roc_auc)

Bi-LSTM Results
----------------------
Accuracy : 0.8647485883301963
Precision: 0.8753096614368291
Recall   : 0.8518617733726226
F1 Score : 0.8634265544393158
ROC-AUC  : 0.929971227188084


In [ ]:
bilstm_model.save("../models/bilstm_model.keras")

In [28]:
results.append({
    "Model": "Bi-LSTM",
    "Accuracy": bilstm_accuracy,
    "Precision": bilstm_precision,
    "Recall": bilstm_recall,
    "F1 Score": bilstm_f1,
    "ROC-AUC": bilstm_roc_auc
})

In [29]:
comparison_df = pd.DataFrame(results)
comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN,0.871067,0.875677,0.866059,0.870842,0.943045
1,RNN,0.498924,0.500806,0.499598,0.500201,0.502188
2,LSTM,0.863135,0.862483,0.865256,0.863867,0.931760
3,GRU,0.870799,0.866667,0.877578,0.872088,0.938827
4,Bi-LSTM,0.864749,0.875310,0.851862,0.863427,0.929971


In [ ]:
comparison_df.to_csv("../models/phase4_model_comparison.csv",index=False)

In [30]:
from tensorflow.keras.models import load_model

ann_model = load_model("../models/ann_model.keras")
rnn_model = load_model("../models/rnn_model.keras")
lstm_model = load_model("../models/lstm_model.keras")
gru_model = load_model("../models/gru_model.keras")
bilstm_model = load_model("../models/bilstm_model.keras")